In [13]:
import pandas as pd
import numpy as np
import geopandas as gpd

import tzfpy
import matplotlib.pyplot as plt

from shapely.geometry import Point, Polygon
import src.makePlots as makePlots

import pickle
from pathlib import Path

In [15]:
def load_pickle(filepath: str): 
    with open(filepath, "rb") as f: 
        temp = pickle.load(f)
    return temp

def dump_pickle(filepath: str, var): 
    with open(filepath, "wb") as f: 
        pickle.dump(var, f)
    return

In [ ]:
test = Path(r"G:\My Drive\Roots\NPP\STAQS\data\Surface\hourly_44201_2023\hourly_44201_2023.csv")


WindowsPath('G:/My Drive/Roots/NPP/STAQS/data/Surface/hourly_44201_2023/hourly_44201_2023.parquet')

In [52]:
EPA_Pregenerated_Files = {
    "Ozone":  r"G:\My Drive\Roots\NPP\STAQS\data\Surface\hourly_44201_2023\hourly_44201_2023.csv",
    }

Field_of_Regard = ["-75.772705","39.690281","-69.180908","42.747012"]


for parameter, csv_filepath in EPA_Pregenerated_Files.items(): 
    parquet_filepath = Path(csv_filepath).with_suffix(".parquet")
    
    if not parquet_filepath.is_file() and csv_filepath.is_file():
        pd.read_csv(csv_filepath, low_memory=False).to_parquet(parquet_filepath)

    df = pd.read_parquet(parquet_filepath)
    df["geometry"] =  [Point(lon, lat) for lat, lon in zip(df["Latitude"], df["Longitude"])]
    df = df[["Parameter Name", "Sample Measurement", "Uncertainty", "State Name", "County Name", "Units of Measure", "geometry"]].copy()

In [50]:
df

,Parameter Name,Sample Measurement,Uncertainty,State Name,County Name,Units of Measure
0,Ozone,0.025,NaN,Alabama,Baldwin,Parts per million
1,Ozone,0.024,NaN,Alabama,Baldwin,Parts per million
2,Ozone,0.023,NaN,Alabama,Baldwin,Parts per million
3,Ozone,0.022,NaN,Alabama,Baldwin,Parts per million
4,Ozone,0.022,NaN,Alabama,Baldwin,Parts per million
...,...,...,...,...,...,...
9027604,Ozone,0.003,NaN,Country Of Mexico,SONORA,Parts per million
9027605,Ozone,0.007,NaN,Country Of Mexico,SONORA,Parts per million
9027606,Ozone,0.007,NaN,Country Of Mexico,SONORA,Parts per million
9027607,Ozone,0.006,NaN,Country Of Mexico,SONORA,Parts per million


In [25]:
df["geometry"] = [Point(lon, lat) for lat, lon in zip(test["Latitude"], test["Longitude"])]

In [26]:
df["datetime"] = pd.to_datetime(test["Date Local"] + " " + test["Time Local"])

In [27]:
df.set_index("datetime", inplace=True)

In [28]:
df = gpd.GeoDataFrame(df, geometry="geometry")

In [29]:
df.head()

,Parameter Name,Sample Measurement,Uncertainty,State Name,County Name,Units of Measure,geometry
datetime,,,,,,,
2023-03-01 01:00:00,Ozone,0.025,NaN,Alabama,Baldwin,Parts per million,POINT (-87.88026 30.49748)
2023-03-01 02:00:00,Ozone,0.024,NaN,Alabama,Baldwin,Parts per million,POINT (-87.88026 30.49748)
2023-03-01 03:00:00,Ozone,0.023,NaN,Alabama,Baldwin,Parts per million,POINT (-87.88026 30.49748)
2023-03-01 04:00:00,Ozone,0.022,NaN,Alabama,Baldwin,Parts per million,POINT (-87.88026 30.49748)
2023-03-01 05:00:00,Ozone,0.022,NaN,Alabama,Baldwin,Parts per million,POINT (-87.88026 30.49748)


In [30]:
locations = df.drop_duplicates(subset="geometry", keep="first")

In [32]:
from shapely.geometry import box

polygon = box("-75.772705","39.690281","-69.180908","42.747012")
midatlantic =  locations.clip(polygon)

In [ ]:
makePlots.site_map(
    {"EPA Ozone Sites": midatlantic}
    )